[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

# 📝 Ejercicio Guiado — Predicción de Precios de Laptops con Ensambles

**Nombre:** ___________________________  
**Fecha:** ___________________________

---

En este ejercicio construirás modelos de **Random Forest Regressor** y **Gradient Boosting Regressor** para predecir el precio en euros de laptops a partir de sus especificaciones técnicas.

El dataset proviene de Kaggle:  
📦 [Laptop Price Dataset](https://www.kaggle.com/datasets/muhammetvarl/laptop-price)

Seguirás los mismos pasos del notebook de referencia (Penguins RF & GB), pero adaptados a **regresión** y con las siguientes particularidades:

| Paso | Notebook de referencia | **Este ejercicio** |
|---|---|---|
| Tarea | Clasificación | **Regresión** |
| División | Train / Test | **Train / Validation / Test** |
| Búsqueda de hiperparámetros | GridSearchCV en train | **GridSearchCV en train, validación manual en val** |
| Escalado | StandardScaler | **MinMaxScaler** |
| Encoding | OneHotEncoder | **OneHotEncoder + OrdinalEncoder según la variable** |
| EDA | Básico | **Correlaciones, histograma target, boxplots + interpretación** |
| Predicción final | — | **Muestra sintética propia** |

### Columnas del dataset

| Columna | Tipo | Descripción |
|---|---|---|
| `Company` | Categórica | Fabricante (Dell, Apple, HP...) |
| `TypeName` | Categórica | Tipo de laptop (Notebook, Gaming, Ultrabook...) |
| `Inches` | Numérica | Tamaño de pantalla en pulgadas |
| `ScreenResolution` | Categórica | Resolución de pantalla |
| `Cpu` | Categórica | Procesador |
| `Ram` | Categórica/Texto | RAM (ej. "8GB") |
| `Memory` | Categórica/Texto | Almacenamiento |
| `Gpu` | Categórica | Tarjeta gráfica |
| `OpSys` | Categórica | Sistema operativo |
| `Weight` | Categórica/Texto | Peso (ej. "1.37kg") |
| `Price_euros` | Numérica 🎯 | **Variable objetivo** — Precio en euros |

## 0. Descarga del dataset

**Opción A – Kaggle API:**
```bash
!pip install kaggle -q
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d muhammetvarl/laptop-price --unzip
```

**Opción B – Descarga manual:**  
Descarga `laptop_price.csv` desde Kaggle y súbelo a la misma carpeta del notebook.


## 1. Librerías

Importa todas las librerías que necesitarás durante el ejercicio.

**Pistas — qué importar y de dónde:**
- `pandas`, `numpy` y `matplotlib.pyplot` para datos y visualizaciones
- De `sklearn.ensemble`: `RandomForestRegressor` y `GradientBoostingRegressor`
- De `sklearn.model_selection`: `train_test_split` y `GridSearchCV`
- De `sklearn.pipeline`: `Pipeline`
- De `sklearn.compose`: `ColumnTransformer`
- De `sklearn.preprocessing`: `RobustScaler`, `OneHotEncoder`, `OrdinalEncoder`
- De `sklearn.impute`: `SimpleImputer`
- De `sklearn.metrics`: `mean_absolute_error`, `mean_squared_error`, `r2_score`
- Define `random_state = 42` para reproducibilidad

> 💡 **¿Qué es RobustScaler?**  
> Escala usando la **mediana** y el **rango intercuartílico (IQR)** en lugar de la media y desviación estándar.  
> Fórmula: `x_scaled = (x − mediana) / IQR`  
> Es más **resistente a outliers** que StandardScaler o MinMaxScaler, lo que lo hace ideal cuando los datos de precios tienen valores extremos.

In [ ]:
# Importa aquí todas las librerías necesarias:


# Define random_state para reproducibilidad:
random_state = 42

## 2. Carga y limpieza de datos

Carga el dataset y realiza las transformaciones necesarias para poder usarlo.

**Pistas:**
- Usa `pd.read_csv('laptop_price.csv')` para cargar
- Revisa `.shape`, `.head()`, `.info()` e `.isnull().sum()`
- **Limpieza importante:** algunas columnas numéricas vienen como texto:
  - `Ram` contiene valores como `'8GB'` → extrae el número con `.str.replace('GB','').astype(int)`
  - `Weight` contiene valores como `'1.37kg'` → extrae el número con `.str.replace('kg','').astype(float)`
- Verifica con `.dtypes` que `Ram` e `Inches` sean numéricas después de la conversión
- Revisa si hay valores nulos y decide qué hacer con ellos

In [ ]:
# Carga el dataset:


In [ ]:
# Revisa tipos de datos, dimensiones y valores nulos:


In [ ]:
# Limpia las columnas Ram y Weight (elimina el sufijo textual y convierte a numérico):


In [ ]:
# Verifica que los tipos de dato sean correctos después de la limpieza:


## 3. Análisis Exploratorio (EDA)

Antes de modelar, es fundamental entender los datos. Realiza los siguientes análisis y escribe una interpretación debajo de cada gráfica.

### 3a. Distribución del target (`Price_euros`)
**Pistas:**
- Crea un histograma con `plt.hist(df['Price_euros'], bins=40, ...)`
- ¿La distribución es simétrica o tiene sesgo? ¿Hay outliers evidentes?
- Calcula también la media, mediana y desviación estándar del precio

### 3b. Correlación de variables numéricas con el target
**Pistas:**
- Selecciona solo las columnas numéricas con `.select_dtypes(include='number')`
- Calcula la correlación con `.corr()['Price_euros'].drop('Price_euros').sort_values()`
- Grafica las correlaciones como barras horizontales
- ¿Qué variables numéricas están más correlacionadas con el precio?

### 3c. Boxplots de variables categóricas clave
**Pistas:**
- Crea boxplots de `Price_euros` agrupados por las columnas: `Company`, `TypeName`, `Ram`, `OpSys`
- Usa `df.boxplot(column='Price_euros', by='Company', ...)` o matplotlib manual
- ¿Qué fabricante tiene precios más altos? ¿Qué tipo de laptop es más cara?

In [ ]:
# 3a. Histograma del target (Price_euros):
# Pista: usa plt.hist(..., bins=40, edgecolor='white')
# Agrega líneas verticales para la media (roja) y la mediana (verde)
# Pista: plt.axvline(df['Price_euros'].mean(), color='red', linestyle='--', label='Media')



**📝 Interpretación del histograma:**  
*Escribe aquí tu análisis. Responde: ¿la distribución es simétrica o tiene sesgo? ¿La media o la mediana es más representativa? ¿Hay laptops extremadamente caras que podrían ser outliers?*

*(Tu respuesta aquí...)*

In [ ]:
# 3b. Correlación de variables numéricas con Price_euros:
# Pista:
#   num_df = df.select_dtypes(include='number')
#   corr = num_df.corr()['Price_euros'].drop('Price_euros').sort_values()
#   plt.barh(corr.index, corr.values, color=[...])



**📝 Interpretación del gráfico de correlaciones:**  
*¿Qué variable numérica tiene mayor correlación positiva con el precio? ¿Tiene sentido intuitivo? ¿Alguna variable sorpresivamente baja o negativa?*

*(Tu respuesta aquí...)*

In [ ]:
# 3c. Boxplot de Price_euros por TypeName:
# Pista: df.boxplot(column='Price_euros', by='TypeName', figsize=(10,5), rot=30)
# Ajusta el título con plt.title(...) y elimina el título automático con plt.suptitle('')



**📝 Interpretación del boxplot por TypeName:**  
*¿Qué tipo de laptop tiende a ser más caro? ¿Cuál tiene mayor dispersión de precios? ¿Hay outliers notables en algún tipo?*

*(Tu respuesta aquí...)*

In [ ]:
# 3c. Boxplot de Price_euros por Company:



**📝 Interpretación del boxplot por Company:**  
*¿Qué fabricante tiene los precios más elevados en mediana? ¿Y el rango más amplio? ¿Coincide con tu percepción del mercado de laptops?*

*(Tu respuesta aquí...)*

In [ ]:
# 3c. Boxplot de Price_euros por Ram:
# Pista: ordena los valores de Ram numéricamente antes de graficar
# df_sorted = df.copy(); df_sorted['Ram'] = ...



**📝 Interpretación del boxplot por Ram:**  
*¿A más RAM, mayor precio? ¿La relación es lineal o hay saltos? ¿Qué implicación tiene esto para el modelo?*

*(Tu respuesta aquí...)*

## 4. División de datos: Train / Validation / Test

A diferencia del notebook de referencia que usa solo Train/Test, aquí dividirás los datos en **tres conjuntos**:

| Conjunto | Proporción | Uso |
|---|---|---|
| **Train** | 60% | Entrenamiento y GridSearchCV |
| **Validation** | 20% | Comparar modelos y elegir el mejor |
| **Test** | 20% | Evaluación final — ¡solo tocar al final! |

**Pistas:**
- Primero separa la variable objetivo (`Price_euros`) de las features
- Haz un primer `train_test_split` para separar test (20%) del resto (80%)
- Luego haz un segundo `train_test_split` sobre el 80% para obtener train (75% del 80% = 60% total) y validation (25% del 80% = 20% total)
- Usa `random_state=random_state` en ambas divisiones
- Imprime el tamaño de cada conjunto para verificar las proporciones

> 💡 **¿Por qué tres conjuntos?**  
> Si usas el test set para elegir entre modelos, contaminas la evaluación final.  
> El validation set te permite comparar modelos honestamente sin «quemar» el test set.

In [ ]:
# Separa X e y:
TARGET = 'Price_euros'
# X = ...
# y = ...


# Primera división: separa el test set (20%):
# X_temp, X_test, y_temp, y_test = train_test_split(..., test_size=0.2, random_state=random_state)


# Segunda división: del 80% restante, separa validation (25% = 20% del total):
# X_train, X_val, y_train, y_val = train_test_split(..., test_size=0.25, random_state=random_state)


# Verifica los tamaños:


## 5. Identificación de columnas por tipo

Identifica qué columnas son numéricas y cuáles son categóricas para aplicar transformaciones distintas.

**Pistas:**
- Usa `X_train.select_dtypes(include='number').columns` para numéricas
- Usa `X_train.select_dtypes(include='object').columns` para categóricas
- Imprime ambas listas para verificar

> ⚠️ **Importante:** las columnas `Cpu`, `Gpu`, `Memory` y `ScreenResolution` tienen **muchos valores únicos**. En el paso de preprocesamiento tendrás que decidir si aplicarles OneHotEncoder (que crearía muchas columnas) u OrdinalEncoder (que las trata como si tuvieran orden). Piensa cuál tiene más sentido para cada caso.

In [ ]:
# Identifica las columnas numéricas:


# Identifica las columnas categóricas:


# Para las categóricas, revisa cuántos valores únicos tiene cada una:
# Pista: X_train[cat_cols].nunique()


## 6. Preprocesamiento con ColumnTransformer

Construye un `ColumnTransformer` con las transformaciones apropiadas para cada tipo de columna.

### Pipeline numérico
Pasos a aplicar en orden:
1. `SimpleImputer(strategy='median')` — imputa nulos con la mediana
2. `MinMaxScaler()` — escala usando max y min

### Pipeline categórico — columnas con pocos valores únicos
Para columnas como `Company`, `TypeName`, `OpSys` (≤ 10 valores únicos aprox.):
1. `SimpleImputer(strategy='most_frequent')`
2. `OneHotEncoder(handle_unknown='ignore', sparse_output=False)`

### Pipeline categórico — columnas con muchos valores únicos
Para columnas como `Cpu`, `Gpu`, `Memory`, `ScreenResolution` (muchos valores únicos):  
1. `SimpleImputer(strategy='most_frequent')`
2. `OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)` — asigna un número entero a cada categoría

> 💡 **¿Por qué no OneHotEncoder para columnas con muchos valores únicos?**  
> OneHotEncoder crea **una columna por cada valor único**. Si `Cpu` tiene 100 valores distintos, generaría 100 columnas nuevas, lo cual aumenta la dimensionalidad innecesariamente y puede perjudicar el modelo.  
> Los modelos de árbol (RF, GB) pueden manejar bien variables ordinales incluso cuando el orden no tiene sentido real.

**Pistas:**
- Crea tres listas de columnas: `num_cols`, `cat_ohe_cols`, `cat_ord_cols`
- Crea tres pipelines separados
- Combínalos en un `ColumnTransformer` con tres transformers
- Recuerda que `sparse_output=False` en `OneHotEncoder` devuelve un array denso (más fácil de manejar)

In [ ]:
# Define las tres listas de columnas:
# num_cols     = columnas numéricas
# cat_ohe_cols = categóricas con pocos valores únicos → OneHotEncoder
# cat_ord_cols = categóricas con muchos valores únicos → OrdinalEncoder



In [ ]:
# Crea el pipeline numérico (SimpleImputer → RobustScaler):


# Crea el pipeline categórico para OHE (SimpleImputer → OneHotEncoder):


# Crea el pipeline categórico para Ordinal (SimpleImputer → OrdinalEncoder):


# Combina los tres en un ColumnTransformer:



## 7. Definición de modelos y búsqueda de hiperparámetros

Define los dos pipelines (RF y GB) y la grilla de hiperparámetros para `GridSearchCV`.

**Pistas para los pipelines:**
```python
pipeline_rf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(random_state=random_state))
])
```

**Pistas para la grilla de hiperparámetros:**  
Nota que el prefijo es `regressor__` (no `classifier__` como en el notebook de referencia):
```python
param_grid = {
    'regressor__n_estimators': [50, 100],
    'regressor__max_depth': [3, 5, 10, None],
    'regressor__min_samples_leaf': [1, 5, 10]
}
```

**Pistas para GridSearchCV:**
- Usa `cv=3` para validación cruzada de 3 folds
- Usa `scoring='neg_mean_absolute_error'` (negativo porque GridSearchCV maximiza)
- Usa `n_jobs=-1` para paralelizar y acelerar la búsqueda
- Crea un `GridSearchCV` para el RF y otro para el GB

> 💡 **¿Por qué `neg_mean_absolute_error`?**  
> `GridSearchCV` siempre **maximiza** el score. Como el MAE es un error que queremos minimizar, se usa el valor negativo: maximizar `-MAE` equivale a minimizar `MAE`.

In [ ]:
# Define el pipeline para Random Forest:


# Define el pipeline para Gradient Boosting:


# Define la grilla de hiperparámetros:


# Define GridSearchCV para Random Forest:


# Define GridSearchCV para Gradient Boosting:



## 8. Entrenamiento

Entrena ambos modelos sobre los datos de entrenamiento. Puede tardar unos minutos.

**Pistas:**
- Usa `rf_search.fit(X_train, y_train)` y `gb_search.fit(X_train, y_train)`
- Después del entrenamiento, muestra los mejores hiperparámetros con `.best_params_`
- Muestra también el mejor score de CV con `.best_score_`

> ⚠️ Si el entrenamiento tarda demasiado, reduce la grilla a menos combinaciones (por ejemplo, solo 2 valores por hiperparámetro).

In [ ]:
# Entrena el Random Forest (GridSearchCV):


# Entrena el Gradient Boosting (GridSearchCV):



In [ ]:
# Muestra los mejores hiperparámetros encontrados para Random Forest:


# Muestra los mejores hiperparámetros encontrados para Gradient Boosting:



## 9. Evaluación de métricas

Evalúa ambos modelos en los tres conjuntos de datos y compara sus resultados.

**Métricas a calcular para cada modelo y cada conjunto (train, val, test):**
- **MAE** — `mean_absolute_error(y_true, y_pred)` — error promedio en euros
- **RMSE** — `sqrt(mean_squared_error(y_true, y_pred))` — penaliza errores grandes
- **R²** — `r2_score(y_true, y_pred)` — proporción de varianza explicada

**Pistas:**
- Crea una función `evaluar(nombre, modelo, X, y)` que reciba el modelo y los datos y retorne (o imprima) las tres métricas
- Llama a la función para (RF en train), (RF en val), (GB en train), (GB en val)
- **Solo al final**, evalúa en test el modelo que mejor haya funcionado en validación
- Organiza los resultados en una tabla con `pd.DataFrame`

> 💡 **Señales de sobreajuste:**  
> Si el R² en train es 0.98 pero en val es 0.65 → el modelo memorizó el train.  
> Un buen modelo tiene métricas similares en train, val y test.

In [ ]:
# Crea una función que calcule MAE, RMSE y R² dado un modelo y un conjunto de datos:
# def evaluar(nombre, modelo, X, y):
#     y_pred = modelo.predict(X)
#     ...



In [ ]:
# Evalúa ambos modelos en train y validación:



In [ ]:
# Elige el mejor modelo según val, y evalúalo en el test set:
# ⚠️ Solo hacer esto una vez al final — no uses test para tomar decisiones



## 10. Visualización de resultados

Genera las siguientes gráficas para el **mejor modelo en el test set**:

**Gráfica 1 — Real vs Predicho:**
- Scatter plot de `y_test` (eje x) vs `y_pred_test` (eje y)
- Agrega la línea diagonal de predicción perfecta en rojo
- Etiqueta los ejes y agrega el R² en el título

**Gráfica 2 — Distribución de residuos:**
- Histograma de `y_test - y_pred_test`
- Línea vertical en 0
- ¿Los residuos son simétricos alrededor de 0?

**Gráfica 3 — Importancia de features:**
- Accede a las importancias con `modelo.best_estimator_.named_steps['regressor'].feature_importances_`
- Muéstralas como barras horizontales ordenadas de mayor a menor
- Para los nombres de features, recuerda que el ColumnTransformer reorganizó las columnas

> 💡 **Pista para nombres de features con ColumnTransformer:**  
> Usa `preprocessor.get_feature_names_out()` después de hacer `.fit()` para obtener los nombres en el mismo orden que las importancias.

In [ ]:
# Gráfica 1: Real vs Predicho en test set:



In [ ]:
# Gráfica 2: Distribución de residuos en test set:



In [ ]:
# Gráfica 3: Importancia de features del mejor modelo:
# Pista: feature_importances_ está en el paso 'regressor' del pipeline
# best_model.best_estimator_.named_steps['regressor'].feature_importances_



## 11. Reflexión sobre los resultados

Responde estas preguntas aquí (doble clic para editar):

**1. ¿Qué modelo (RF o GB) funcionó mejor en validación? ¿Y en test? ¿Coinciden?**  
*(Tu respuesta aquí...)*

---

**2. Mirando la gráfica de importancia de features, ¿qué variable impacta más el precio? ¿Lo esperabas basándote en el EDA?**  
*(Tu respuesta aquí...)*

---

**3. ¿Hay indicios de sobreajuste? ¿Cómo lo sabes comparando las métricas de train vs val?**  
*(Tu respuesta aquí...)*

---

**4. ¿Para qué sirve el conjunto de validación si ya usamos validación cruzada (GridSearchCV) dentro del train? ¿Son lo mismo?**  
*(Tu respuesta aquí...)*

## 12. 🧪 Prueba con muestra sintética propia

Diseña **al menos 5 laptops ficticias** con perfiles muy diferentes y predice su precio.

La clave es razonar **antes de ejecutar** cuánto debería costar cada laptop según sus especificaciones.

**Perfiles sugeridos:**

| # | Perfil | Características esperadas |
|---|---|---|
| 1 | Laptop gama alta de trabajo | i7, 32GB RAM, MacBook |
| 2 | Laptop gaming potente | i7, 16GB RAM, GPU dedicada |
| 3 | Laptop básica de oficina | i3, 4GB RAM, Windows |
| 4 | Ultrabook premium | i5, 8GB RAM, 13 pulgadas |
| 5 | Laptop económica sin SO | AMD, 4GB RAM, No OS |

**Pistas importantes:**
- El DataFrame debe tener **exactamente las mismas columnas** que `X_train`
- Los valores de columnas categóricas deben coincidir con los que aparecen en el dataset original (usa `.unique()` para revisar valores válidos)
- `Ram` debe ser numérico (sin 'GB'), `Weight` debe ser numérico (sin 'kg') — ¡ya que limpiaste esas columnas!
- Usa `mejor_modelo.predict(muestra)` para obtener las predicciones

In [ ]:
# Revisa los valores válidos de las columnas categóricas para no cometer errores:
# Pista: for col in cat_ohe_cols + cat_ord_cols: print(col, X_train[col].unique())



In [ ]:
# Diseña tus 5 laptops ficticias.
# ANTES de ejecutar, escribe en comentarios cuánto esperas que cueste cada una:

# Laptop 1 (perfil: ...): precio esperado ~€___
# Laptop 2 (perfil: ...): precio esperado ~€___
# Laptop 3 (perfil: ...): precio esperado ~€___
# Laptop 4 (perfil: ...): precio esperado ~€___
# Laptop 5 (perfil: ...): precio esperado ~€___

muestra = pd.DataFrame({
    'Company':          [],   # Completa con valores válidos
    'TypeName':         [],
    'Inches':           [],
    'ScreenResolution': [],
    'Cpu':              [],
    'Ram':              [],   # Número sin 'GB', ej: 8
    'Memory':           [],
    'Gpu':              [],
    'OpSys':            [],
    'Weight':           []    # Número sin 'kg', ej: 1.5
})

muestra

In [ ]:
# Genera las predicciones de precio para tus laptops:
# Pista: predicciones = mejor_modelo.predict(muestra)


# Muestra los resultados en una tabla legible:
# Pista: muestra_result = muestra.copy()
#        muestra_result['Precio Predicho (€)'] = predicciones.round(2)



### Reflexión sobre la muestra sintética

Responde aquí (doble clic para editar):

**1. ¿Las predicciones se acercan a lo que esperabas? ¿Cuál te sorprendió más y por qué?**  
*(Tu respuesta aquí...)*

---

**2. ¿El modelo diferencia bien entre una laptop básica y una gama alta? ¿La diferencia de precio predicha te parece razonable?**  
*(Tu respuesta aquí...)*

---

**3. Prueba cambiar solo el fabricante (`Company`) manteniendo todo lo demás igual. ¿Cuánto cambia el precio predicho? ¿Tiene sentido?**  
*(Tu respuesta aquí...)*